## args_schema

In [5]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field


class WeatherSchema(BaseModel):
    city: str = Field(
        description="城市名",
        default="Shanghai"
    )
    is_forecast: bool = Field(
        description="是否获取明日的预报",
        default=False
    )


@tool(name_or_callable="get_weather_and_forecast",
      description="获取今日的天气和明日的天气预报",
      args_schema=WeatherSchema)
def get_weather(city: str, is_forecast: bool):
    return f"{city}今天暴雨" + "，明天雷阵雨" if is_forecast else ""


load_dotenv(override=True)

model = init_chat_model(model="deepseek:deepseek-v4-flash")
model_with_tools = model.bind_tools([get_weather])
messages: list[BaseMessage] = [HumanMessage(content="浦东明天和今天的天气如何？")]

response = model_with_tools.invoke(input=messages)
messages.append(response)

tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call.get("name") == "get_weather_and_forecast":
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

final_response = model_with_tools.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

浦东明天和今天的天气如何？
================================== Ai Message ==================================

好的，我来查询浦东（上海）今天和明天的天气情况。
Tool Calls:
  get_weather_and_forecast (call_00_zz1El7BuTMtCC3BNxoex0848)
 Call ID: call_00_zz1El7BuTMtCC3BNxoex0848
  Args:
    city: 上海
    is_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

上海今天暴雨明天雷阵雨
================================== Ai Message ==================================

根据查询结果，浦东（上海）的天气情况如下：

### 📅 今天（当前）
- **天气：暴雨** 🌧️🌧️🌧️
- 出门记得带好雨具，注意出行安全，暴雨天气能见度较低。

### 📅 明天（预报）
- **天气：雷阵雨** ⛈️
- 依然有雨，且可能伴有雷电，外出时注意防雷避雨。

**总结：** 今明两天浦东都有明显降雨，今天雨势更大（暴雨），明天转为雷阵雨。建议非必要减少外出，如需出行请携带雨具并注意安全！☂️


## docstring

In [9]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.tools import tool


@tool(name_or_callable="get_weather_and_forecast", parse_docstring=True)
def get_weather(city: str = "Shanghai", is_forecast: bool = False):
    """
    获取今日的天气和明日的天气预报

    Args:
        city: 城市名
        is_forecast: 是否获取明日的预报
    """
    return f"{city}今天暴雨" + ("，明天雷阵雨" if is_forecast else "")


load_dotenv(override=True)

model = init_chat_model(model="deepseek:deepseek-v4-flash")
model_with_tools = model.bind_tools([get_weather])
messages: list[BaseMessage] = [HumanMessage(content="浦东明天和今天的天气如何？")]

response = model_with_tools.invoke(input=messages)
messages.append(response)

tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call.get("name") == "get_weather_and_forecast":
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

final_response = model_with_tools.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

浦东明天和今天的天气如何？
================================== Ai Message ==================================

好的，我来查一下浦东今天和明天的天气情况。
Tool Calls:
  get_weather_and_forecast (call_00_mjHiTerTHWA1eyarKNwX9499)
 Call ID: call_00_mjHiTerTHWA1eyarKNwX9499
  Args:
    city: 浦东
    is_forecast: False
  get_weather_and_forecast (call_01_S2k2Sp5bzGADDe1BaEin7523)
 Call ID: call_01_S2k2Sp5bzGADDe1BaEin7523
  Args:
    city: 浦东
    is_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

浦东今天暴雨
================================= Tool Message =================================
Name: get_weather_and_forecast

浦东今天暴雨，明天雷阵雨
================================== Ai Message ==================================

查询结果如下：

### 🌤 浦东新区天气情况

| 日期 | 天气 |
|------|------|
| **今天** | ⛈ **暴雨** |
| **明天** | 🌩 **雷阵雨** |

看来浦东这两天都有明显的降水天气，今天雨势较大（暴雨），明天转为雷阵雨。建议您：

- **今天**：尽量减少外出，出门务必带好

## 多工具调用

In [14]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每条新闻占一行
    """

    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)


# rprint(convert_to_openai_tool(search_news))

# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)
message_list = []
# human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)
# 3.工具调用
while True:
    response = model_with_tools.invoke(message_list)
    message_list.append(response)
    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    # 如果有调用工具，处理工具调用响应
    # 4.开发者根据模型的响应，调用工具并获取结果
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result", stock_result)
            message_list.append(stock_result)
    if tool_call["name"] == "search_news":
        news_result = search_news.invoke(tool_call)
        print("news_result", news_result)
        message_list.append(news_result)

# print("response", response)
# print(response.content)
for msg in message_list:
    msg.pretty_print()

stock_result content='微软公司 today价格: 415.86美元' name='get_stock_price' tool_call_id='call_00_xpga72NDpI047JWpzFTo2946'
stock_result content='苹果公司 today价格: 185.2美元' name='get_stock_price' tool_call_id='call_01_uRX3Jylcqg2ipumRAUGF3381'
没有工具调用，直接返回答案
================================ Human Message =================================

比较一下微软和苹果的股价
================================== Ai Message ==================================

好的，我来同时查询微软和苹果的股票价格信息。
Tool Calls:
  get_stock_price (call_00_xpga72NDpI047JWpzFTo2946)
 Call ID: call_00_xpga72NDpI047JWpzFTo2946
  Args:
    company: 微软公司
  get_stock_price (call_01_uRX3Jylcqg2ipumRAUGF3381)
 Call ID: call_01_uRX3Jylcqg2ipumRAUGF3381
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

微软公司 today价格: 415.86美元
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today价格: 185.2美元
================================== Ai Messag

In [15]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'


@tool(parse_docstring=True)
def get_news() -> str:
    """
    获取当日新闻
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"


model_with_tools = model.bind_tools([get_weather, get_news])
messages = [
    HumanMessage("今天杭州天气如何？今天新闻是什么？别瞎编")
]
response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

好的，我先查一下杭州今天的天气和今天的新闻，稍等！
Tool Calls:
  get_weather (call_00_WJKnvib0vi4kpebfgm9k0275)
 Call ID: call_00_WJKnvib0vi4kpebfgm9k0275
  Args:
    city: 杭州
  get_news (call_01_RhWkT9IYe8fWpcIAesqJ0722)
 Call ID: call_01_RhWkT9IYe8fWpcIAesqJ0722
  Args:


In [16]:
messages.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")

final_response = model.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()

content='杭州当天晴朗' name='get_weather' tool_call_id='call_00_WJKnvib0vi4kpebfgm9k0275'
content='近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。' name='get_news' tool_call_id='call_01_RhWkT9IYe8fWpcIAesqJ0722'
================================ Human Message =================================

今天杭州天气如何？今天新闻是什么？别瞎编
================================== Ai Message ==================================

好的，我先查一下杭州今天的天气和今天的新闻，稍等！
Tool Calls:
  get_weather (call_00_WJKnvib0vi4kpebfgm9k0275)
 Call ID: call_00_WJKnvib0vi4kpebfgm9k0275
  Args:
    city: 杭州
  get_news (call_01_RhWkT9IYe8fWpcIAesqJ0722)
 Call ID: call_01_RhWkT9IYe8fWpcIAesqJ0722
  Args:
================================= Tool Message =================================
Name: get_weather

杭州当天晴朗
================================= Tool Message =================================
Name: get_news

近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。
================================== Ai Message ==================================
